# Async and await
Use asynchronous I/O so one slow request does not block others.


In [ ]:
# While one call waits for I/O, the event loop can run another call.
import asyncio

async def fetch(name: str) -> str:
    await asyncio.sleep(0.01)
    return name.upper()

print(await asyncio.gather(fetch("a"), fetch("b")))


## Polished version
Define an interface for the external provider and let the service coordinate concurrent calls.


In [ ]:
# The service coordinates concurrency without knowing the provider implementation.
from dataclasses import dataclass
from typing import Protocol

@dataclass(frozen=True)
class ChatRequest:
    prompt: str

@dataclass(frozen=True)
class ChatResponse:
    text: str

class LLMProvider(Protocol):
    async def complete(self, request: ChatRequest) -> ChatResponse: ...

class MemoryProvider:
    async def complete(self, request: ChatRequest) -> ChatResponse:
        await asyncio.sleep(0.01)
        return ChatResponse(request.prompt.upper())

class BatchChatService:
    def __init__(self, provider: LLMProvider) -> None:
        self.provider = provider

    async def complete_all(self, prompts: list[str]) -> list[ChatResponse]:
        requests = [ChatRequest(prompt) for prompt in prompts]
        # gather starts calls together and preserves result order.
        return await asyncio.gather(*(self.provider.complete(item) for item in requests))

service = BatchChatService(MemoryProvider())
print(await service.complete_all(["hello", "world"]))
